# ACME Inc. - Team Structure & Performance

**Answering seven business questions from the Gold layer of the data lake.**

Information about team structure and performance was scattered across six source
systems. This notebook reads the curated Gold layer produced by the batch
pipeline and answers each question with a number, a chart and a plain-English
interpretation.

Every figure here is read from Gold. Nothing is recalculated in the notebook -
the numbers are computed in the pipeline, versioned, and reproducible, so this
notebook is presentation rather than calculation.

---

### Pipeline overview

### The questions

1. Who are the members of each team?
2. Where are the teams located?
3. What are the key achievements of each team on a monthly basis?
4. How many teams have a team leader not co-located with team members?
5. How many teams have a team leader who is non-direct staff?
6. How many teams have a non-direct staff to employees ratio above 20%?
7. How many teams are reporting to an organisation leader?

## Setup

In [ ]:
import os
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

plt.rcParams["figure.figsize"] = (10, 4.5)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
plt.rcParams["font.size"] = 10

ACCENT = "#2563eb"
MUTED = "#94a3b8"
WARN = "#dc2626"

# Resolve the Gold layer from the same configuration the pipeline uses, so the
# notebook works unchanged against a local lake or an S3 lake.
GOLD = Path(os.environ.get("GOLD_DIR", Path.cwd().parents[2] / "data" / "gold"))
print(f"Reading Gold layer from: {GOLD}")
print(f"Exists: {GOLD.exists()}")

In [ ]:
def read_gold(dataset: str) -> pd.DataFrame:
    """Load one curated dataset from the Gold layer."""
    path = GOLD / dataset
    if not path.exists():
        raise FileNotFoundError(
            f"{path} not found. Run the pipeline first:\n"
            f"    cd backend && python -m data_pipeline.main"
        )
    return pd.read_parquet(path)


datasets = [
    "business_answers",
    "team_members",
    "team_locations",
    "monthly_team_achievements",
    "leader_not_colocated",
    "leader_non_direct_staff",
    "staff_ratio_analysis",
    "organization_reporting_summary",
    "employee_summary",
]

data = {name: read_gold(name) for name in datasets}

summary = pd.DataFrame(
    [{"dataset": n, "rows": len(df), "columns": len(df.columns)} for n, df in data.items()]
)
summary

### Headline answers

The pipeline computes each answer and stores it in `gold.business_answers`, so
the numbers below are exactly what the batch job produced - not a recalculation
that might drift from it.

In [ ]:
answers = data["business_answers"].sort_values("question_id").reset_index(drop=True)

for _, row in answers.iterrows():
    print(f"{row['question_id']}  {row['answer']:>8,}  {row['unit']}")
    print(f"      {row['question']}")
    print(f"      {row['detail']}\n")

---
## Q1 - Who are the members of each team?

The roster combines the team membership export with the unified person
dimension, so every member carries their name, role, location and - critically -
whether they are direct staff or a contractor.

In [ ]:
members = data["team_members"]

print(f"Teams with a roster : {members['team_id'].nunique():,}")
print(f"Membership records  : {len(members):,}")
print(f"Distinct people     : {members['employee_email'].nunique():,}")
print(f"Average team size   : {len(members) / members['team_id'].nunique():.1f}")

example_team = members["team_id"].iloc[0]
example = members[members["team_id"] == example_team][
    ["full_name", "employee_email", "role", "staff_type", "person_city", "allocation_pct", "is_active"]
].sort_values(["role", "full_name"])

print(f"\nExample roster: {members[members['team_id'] == example_team]['team_name'].iloc[0]} ({example_team})")
example

In [ ]:
fig, axes = plt.subplots(1, 2)

sizes = members.groupby("team_id").size()
axes[0].hist(sizes, bins=range(int(sizes.min()), int(sizes.max()) + 2), color=ACCENT, edgecolor="white")
axes[0].set_title("Team size distribution")
axes[0].set_xlabel("members per team")
axes[0].set_ylabel("teams")

composition = members["staff_type"].value_counts()
axes[1].bar(composition.index, composition.values,
            color=[ACCENT if i == "DIRECT" else MUTED for i in composition.index])
axes[1].set_title("Workforce composition across all rosters")
axes[1].set_ylabel("people")
for i, v in enumerate(composition.values):
    axes[1].text(i, v, f"{v:,}", ha="center", va="bottom")

plt.tight_layout()
plt.show()

---
## Q2 - Where are the teams located?

A team has two notions of location: the office it is registered against, and
where its people actually sit. Both are reported, because they frequently
disagree - and that disagreement is the substance of question 4.

In [ ]:
locations = data["team_locations"]

print(f"Distinct offices in use  : {locations['primary_office'].nunique()}")
print(f"Teams spanning >1 location: {int(locations['is_distributed'].sum()):,} "
      f"({locations['is_distributed'].mean():.1%})")
print(f"Office matches where people actually are: "
      f"{int(locations['office_matches_members'].fillna(False).sum()):,}")

locations.groupby(["office_region", "office_city"]).agg(
    teams=("team_id", "count"),
    members=("member_count", "sum"),
).sort_values("teams", ascending=False)

In [ ]:
fig, axes = plt.subplots(1, 2)

by_city = locations["office_city"].value_counts().sort_values()
axes[0].barh(by_city.index, by_city.values, color=ACCENT)
axes[0].set_title("Teams per registered office")
axes[0].set_xlabel("teams")

by_region = locations["office_region"].value_counts()
axes[1].bar(by_region.index, by_region.values, color=ACCENT)
axes[1].set_title("Teams per region")
axes[1].set_ylabel("teams")
for i, v in enumerate(by_region.values):
    axes[1].text(i, v, f"{v:,}", ha="center", va="bottom")

plt.tight_layout()
plt.show()

---
## Q3 - What are the key achievements of each team, monthly?

Achievements are aggregated to one row per team per month. Impact scores arrived
in mixed form: most numeric, some as Low / Medium / High labels, some missing.
Labels were mapped to band midpoints in the Silver layer, and the average over
**numeric scores only** is reported alongside the average over all, so the
estimate never hides inside the headline figure.

In [ ]:
achievements = data["monthly_team_achievements"]

print(f"Months covered      : {achievements['month_key'].nunique()} "
      f"({achievements['month_key'].min()} to {achievements['month_key'].max()})")
print(f"Team-month rows     : {len(achievements):,}")
print(f"Total achievements  : {int(achievements['achievement_count'].sum()):,}")
print(f"Mean impact (numeric only): {achievements['avg_impact_score_numeric'].mean():.2f}")

monthly = achievements.groupby("month_key").agg(
    achievements=("achievement_count", "sum"),
    teams_reporting=("team_id", "nunique"),
    avg_impact=("avg_impact_score_numeric", "mean"),
).round(2)
monthly

In [ ]:
fig, axes = plt.subplots(1, 2)

axes[0].plot(monthly.index, monthly["achievements"], marker="o", color=ACCENT, linewidth=2)
axes[0].set_title("Achievements reported per month")
axes[0].set_ylabel("achievements")
axes[0].tick_params(axis="x", rotation=45)

axes[1].plot(monthly.index, monthly["avg_impact"], marker="o", color=ACCENT, linewidth=2)
axes[1].set_title("Average impact score per month (numeric only)")
axes[1].set_ylabel("impact score")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
top = (
    achievements.groupby(["team_id", "team_name"])
    .agg(achievements=("achievement_count", "sum"),
         total_impact=("total_impact_score", "sum"))
    .sort_values("total_impact", ascending=False)
    .head(10)
    .reset_index()
)
top

---
## Q4 - How many teams have a leader not co-located with their team?

**Definition used:** the leader's own location differs from the location where
most of the team sits. The leader's location comes from the person dimension
rather than the roster, because a leader does not always appear on the roster of
the team they lead.

The alternative reading - leader versus the team's *registered office* - is also
computed, since the two can disagree.

In [ ]:
colocation = data["leader_not_colocated"]

not_colocated = int(colocation["leader_not_colocated"].fillna(False).sum())
office_based = int((colocation["is_colocated_with_office"] == False).sum())
unknown = int(colocation["is_colocated_with_members"].isna().sum())

print(f"ANSWER: {not_colocated:,} teams have a leader not co-located with their team")
print(f"        out of {len(colocation):,} teams ({not_colocated / len(colocation):.1%})\n")
print(f"Alternative definition (vs registered office): {office_based:,} teams")
print(f"Teams where co-location could not be determined: {unknown:,}")

In [ ]:
fig, axes = plt.subplots(1, 2)

counts = [len(colocation) - not_colocated - unknown, not_colocated, unknown]
labels = ["Co-located", "Not co-located", "Undetermined"]
axes[0].bar(labels, counts, color=[ACCENT, WARN, MUTED])
axes[0].set_title("Leader co-location with team members")
axes[0].set_ylabel("teams")
for i, v in enumerate(counts):
    axes[0].text(i, v, f"{v:,}", ha="center", va="bottom")

comparison = {
    "vs team members": not_colocated,
    "vs registered office": office_based,
}
axes[1].bar(comparison.keys(), comparison.values(), color=[WARN, MUTED])
axes[1].set_title("Both definitions compared")
axes[1].set_ylabel("teams not co-located")
for i, v in enumerate(comparison.values()):
    axes[1].text(i, v, f"{v:,}", ha="center", va="bottom")

plt.tight_layout()
plt.show()

---
## Q5 - How many teams have a leader who is non-direct staff?

A person is **direct staff** if they resolve to the employee directory, and
**non-direct** if they resolve only to the vendor roster. This classification is
defined once, in the Silver layer, and every downstream question reads the same
column rather than re-deriving it.

In [ ]:
leaders = data["leader_non_direct_staff"]

non_direct = int(leaders["leader_is_non_direct"].sum())
unresolved = int((leaders["leader_staff_type"] == "UNRESOLVED").sum())

print(f"ANSWER: {non_direct:,} teams are led by non-direct staff")
print(f"        out of {len(leaders):,} teams ({non_direct / len(leaders):.1%})\n")
print(f"Leaders that could not be resolved to any person record: {unresolved:,}")

agencies = (
    leaders[leaders["leader_is_non_direct"]]["leader_agency"]
    .value_counts()
    .rename_axis("agency")
    .reset_index(name="teams_led")
)
agencies

In [ ]:
fig, axes = plt.subplots(1, 2)

types = leaders["leader_staff_type"].value_counts()
colors = [WARN if t == "NON_DIRECT" else (MUTED if t == "UNRESOLVED" else ACCENT) for t in types.index]
axes[0].bar(types.index, types.values, color=colors)
axes[0].set_title("Team leaders by staff type")
axes[0].set_ylabel("teams")
for i, v in enumerate(types.values):
    axes[0].text(i, v, f"{v:,}", ha="center", va="bottom")

if len(agencies):
    axes[1].barh(agencies["agency"][::-1], agencies["teams_led"][::-1], color=WARN)
    axes[1].set_title("Agencies supplying team leaders")
    axes[1].set_xlabel("teams led")

plt.tight_layout()
plt.show()

---
## Q6 - How many teams have a non-direct staff ratio above 20%?

**Definition used:** non-direct staff as a share of all *classified* members -
`non_direct / (direct + non_direct)`. Members whose staff type could not be
resolved are excluded from the denominator so an unresolved person neither
inflates nor deflates the ratio.

The stricter reading of "non-direct **to employees**" - `non_direct / direct` -
is computed alongside it.

In [ ]:
ratios = data["staff_ratio_analysis"]
threshold = ratios["threshold_used"].iloc[0]

above = int(ratios["exceeds_threshold"].sum())

print(f"ANSWER: {above:,} teams exceed the {threshold:.0%} non-direct staff threshold")
print(f"        out of {len(ratios):,} teams ({above / len(ratios):.1%})\n")
print(f"Mean non-direct ratio   : {ratios['non_direct_ratio'].mean():.1%}")
print(f"Median non-direct ratio : {ratios['non_direct_ratio'].median():.1%}")
print(f"Highest ratio observed  : {ratios['non_direct_ratio'].max():.1%}")

ratios.nlargest(10, "non_direct_ratio")[
    ["team_id", "team_name", "member_count", "direct_count", "non_direct_count", "non_direct_ratio"]
]

In [ ]:
fig, axes = plt.subplots(1, 2)

axes[0].hist(ratios["non_direct_ratio"].dropna(), bins=30, color=ACCENT, edgecolor="white")
axes[0].axvline(threshold, color=WARN, linestyle="--", linewidth=2,
                label=f"{threshold:.0%} threshold")
axes[0].set_title("Distribution of non-direct staff ratio")
axes[0].set_xlabel("non-direct share of team")
axes[0].set_ylabel("teams")
axes[0].legend()

split = [len(ratios) - above, above]
axes[1].bar(["At or below threshold", "Above threshold"], split, color=[ACCENT, WARN])
axes[1].set_title(f"Teams above the {threshold:.0%} threshold")
axes[1].set_ylabel("teams")
for i, v in enumerate(split):
    axes[1].text(i, v, f"{v:,}", ha="center", va="bottom")

plt.tight_layout()
plt.show()

---
## Q7 - How many teams report to an organisation leader?

The source carries a `reports_to_type` label, but a label is a claim. Every claim
was checked against the organisation reference data by testing whether the
team's reporting manager really is that organisation's leader. Both the claimed
and the verified counts are published; where they disagree, the verified figure
is the one to quote.

In [ ]:
orgs = data["organization_reporting_summary"]

claimed = int(orgs["teams_claiming_org_leader"].sum())
verified = int(orgs["teams_verified_org_leader"].sum())
unverified = int(orgs["teams_claim_unverified"].sum())

print(f"ANSWER: {verified:,} teams report to an organisation leader (verified)")
print(f"        {claimed:,} teams claim it in the source data")
print(f"        {unverified:,} claims could not be verified\n")
if claimed == verified:
    print("The source label agrees with the reference data in every case.")
else:
    print(f"The source label overstates by {claimed - verified:,} teams.")

orgs[["org_id", "org_name", "team_count", "teams_claiming_org_leader",
      "teams_verified_org_leader", "verified_share"]]

In [ ]:
fig, axes = plt.subplots(1, 2)

x = range(len(orgs))
width = 0.38
axes[0].bar([i - width / 2 for i in x], orgs["team_count"], width, label="all teams", color=MUTED)
axes[0].bar([i + width / 2 for i in x], orgs["teams_verified_org_leader"], width,
            label="report to org leader", color=ACCENT)
axes[0].set_xticks(list(x))
axes[0].set_xticklabels(orgs["org_name"].str.replace(" Organization", ""), rotation=35, ha="right")
axes[0].set_title("Teams reporting to the organisation leader")
axes[0].set_ylabel("teams")
axes[0].legend()

axes[1].bar(orgs["org_id"], orgs["verified_share"], color=ACCENT)
axes[1].set_title("Share of each organisation's teams reporting to its leader")
axes[1].set_ylabel("share")
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()

---
## Data quality: what was found, and what was done about it

Profiling the six source systems surfaced defects in every one of them. The
pipeline handles each explicitly rather than silently, and every rejected record
is written to a quarantine path with the rule it broke, so nothing disappears
without a trace.

| Source defect | Scale | Handling |
| --- | --- | --- |
| Six spellings of full-time employment | 200k rows | Conformed to a single token |
| Four spellings of the corporate email domain | ~12k rows | Repaired to the canonical domain |
| Six different date formats | across all sources | Parsed by trying each in turn |
| `LOC-01` mapped to both Austin and Dallas | 1 row | Collision broken deterministically, loser quarantined |
| Team ids written as `TM-001`, `tm-126` and `tm685` | 46,937 rows | Normalised to one canonical form |
| Duplicate emails in the employee directory | ~49k rows | Resolved by precedence: direct over non-direct, active over inactive |
| Allocation percentages above 100 | ~61k rows | Capped and flagged; original retained |
| Impact scores as Low / Medium / High or missing | ~30k rows | Labels mapped to band midpoints; original form recorded |
| Achievements with no `team_id` | 60,693 rows | Attribution attempted via reporter; the rest quarantined |

### The judgement call that mattered most

Two rules were originally written as errors and rejected valid business records:
allocation above 100%, and contractor engagements ending before they start.
Both would have removed real people from real teams over a typo in an unrelated
field - and the second changed the answer to question 5 by roughly a hundred
teams. Both are now warnings: the record survives, the problem is flagged, and
the classification that the business questions depend on stays intact.

The distinction is **is this record unusable, or merely imperfect?** Only the
first justifies rejection.

In [ ]:
attributed = int(achievements["attributed_via_reporter"].sum())
total_ach = int(achievements["achievement_count"].sum())

print(f"Achievements in Gold                : {total_ach:,}")
print(f"Recovered via reporter attribution  : {attributed:,}")
print("\nAchievements that arrived with no team_id could not be matched on team")
print("name, because 25,000 teams share only 200 distinct names. Matching on name")
print("would have multiplied each unattributed row roughly a hundredfold and")
print("corrupted every downstream metric, so those rows are quarantined instead.")

---
## Assumptions and limitations

**Definitions chosen where the question was ambiguous**

- *Co-located* means the leader's location matches the location where most of the
  team sits. Comparing against the team's registered office gives a different
  number; both are published.
- *Non-direct staff ratio* means non-direct as a share of all classified members.
  The stricter reading, non-direct divided by direct, is published alongside.
- *Direct staff* means the person resolves to the employee directory. Anyone
  appearing in both the directory and the vendor roster is treated as direct,
  since an employment record outranks an agency engagement.

**Known limitations**

- Roughly a third of the achievements feed cannot be attributed to a team. Those
  rows are excluded from question 3 and are recoverable from quarantine if the
  source system is corrected.
- A small number of memberships reference an email that resolves to no person
  record. They remain on the roster flagged `person_found = false` rather than
  being dropped, so team sizes stay accurate even where classification is
  unknown.
- `LOC-01` is genuinely ambiguous in the source. The resolution is deterministic
  and documented, but it is a choice, not a recovery of the true value.
- Impact score band midpoints are estimates. Any average that depends on
  precision should use `avg_impact_score_numeric`, which ignores them.

**Reproducibility**

Every figure in this notebook comes from the Gold layer. Rerunning the pipeline
on the same input reproduces identical output: bronze overwrites a single date
partition, silver and gold overwrite in full, and the achievement surrogate key
is a hash of the payload rather than a generated id.

In [ ]:
checks = {
    "Q1": (int(answers.loc[answers.question_id == "Q1", "answer"].iloc[0]), members["team_id"].nunique()),
    "Q4": (int(answers.loc[answers.question_id == "Q4", "answer"].iloc[0]), not_colocated),
    "Q5": (int(answers.loc[answers.question_id == "Q5", "answer"].iloc[0]), non_direct),
    "Q6": (int(answers.loc[answers.question_id == "Q6", "answer"].iloc[0]), above),
    "Q7": (int(answers.loc[answers.question_id == "Q7", "answer"].iloc[0]), verified),
}

print(f"{'':4} {'pipeline':>10} {'notebook':>10}   match")
for q, (stored, recomputed) in checks.items():
    print(f"{q:4} {stored:>10,} {recomputed:>10,}   {'yes' if stored == recomputed else 'NO'}")

assert all(a == b for a, b in checks.values()), "Notebook disagrees with the pipeline"
print("\nAll headline answers reconcile with gold.business_answers.")